# Silver To Gold Layer

In [2]:
import pandas as pd
pd.set_option('display.max_columns', None)

from datetime import datetime, UTC
import os

import glob

import csv


### Log Function

In [3]:
LOG_FILE = "transformation_log.csv"

# Function to log transformations
def log_transformation(
    input_source: str, # File or table name that is read
    output_source: str, # File or table name that is written to
    action_description: str, # Description of the transformation performed
    performed_by: str, # Name performing the transformation
    log_path: str = LOG_FILE
):
    file_exists = os.path.exists(log_path)

    with open(log_path, mode="a", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)

        if not file_exists:
            writer.writerow([
                "timestamp",
                "input_file_or_table",
                "output_file_or_table",
                "action_description",
                "performed_by"
            ])

        writer.writerow([
            datetime.now(UTC).isoformat(),
            input_source,
            output_source,
            action_description,
            performed_by
        ])


Place Silver Layer Data to Gold Layer

In [21]:
def buildGold():

    # Grab Header
    headerDf = pd.read_csv('SilverLayerData/header.csv')
    headerDf.to_csv('GoldLayerData/fact_header.csv', index=False)
    log_transformation('SilverLayerData/header.csv', 'GoldLayerData/fact_header.csv', 'Grab Data from Silver Layer to Gold', 'Denny Ye')


    # Grab Cargo Measurement
    cargoMeasurementDf = pd.read_csv('SilverLayerData/cargoMeasurement.csv')
    cargoMeasurementDf.to_csv('GoldLayerData/dim_cargo_measurement.csv', index=False)
    log_transformation('SilverLayerData/cargoMeasurement.csv', 'GoldLayerData/dim_cargo_measurement.csv', 'Grab Data from Silver Layer to Gold', 'Denny Ye')

    # Grab Consignee
    consigneeDf = pd.read_csv('SilverLayerData/consignee.csv')
    consigneeDf.to_csv('GoldLayerData/dim_consignee.csv', index=False)
    log_transformation('SilverLayerData/consignee.csv', 'GoldLayerData/dim_consginee.csv', 'Grab Data from Silver Layer to Gold', 'Denny Ye')

    # Grab Port
    portDf = pd.read_csv('SilverLayerData/port.csv')
    portDf.to_csv('GoldLayerData/dim_port.csv', index=False)
    log_transformation('SilverLayerData/port.csv', 'GoldLayerData/dim_port.csv', 'Grab Data from Silver Layer to Gold', 'Denny Ye')

    # Grab Shipper
    shipperDf = pd.read_csv('SilverLayerData/shipper.csv')
    shipperDf.to_csv('GoldLayerData/dim_shipper.csv', index=False)
    log_transformation('SilverLayerData/shipper.csv', 'GoldLayerData/dim_shipper.csv', 'Grab Data from Silver Layer to Gold', 'Denny Ye')

    # Grab Vessel
    vesselDf = pd.read_csv('SilverLayerData/vessel.csv')
    vesselDf.to_csv('GoldLayerData/dim_vessel.csv', index=False)
    log_transformation('SilverLayerData/vessel.csv', 'GoldLayerData/dim_vessel.csv', 'Grab Data from Silver Layer to Gold', 'Denny Ye')

In [22]:
buildGold()

C:\Users\EF782XL\AppData\Local\Temp\ipykernel_20816\3753059278.py:4: DtypeWarning: Columns (0: in_bond_entry_type) have mixed types. Specify dtype option on import or set low_memory=False.
  headerDf = pd.read_csv('SilverLayerData/header.csv')


---

# Call CSV Tables that need cleaning

In [35]:
factHeaderDf = pd.read_csv('GoldLayerData/fact_header.csv')
dimCargoMeasurementDf = pd.read_csv('GoldLayerData/dim_cargo_measurement.csv')

In [36]:
factHeaderDf

,identifier,estimated_arrival_date,actual_arrival_date,port_of_unlading_id,foreign_port_of_lading_qualifier_id,foreign_port_of_lading_id,port_of_destination_id,foreign_port_of_destination_qualifier_id,foreign_port_of_destination_id,vessel_id,shipper_id,consignee_id,manifest_quantity,weight,measurement
0,201901012,2018-03-28,2018-03-30,887,1269,1176,NaN,NaN,NaN,27388,NaN,NaN,3350,44599,12756
1,201901013,2018-03-28,2018-03-30,887,1269,207,NaN,NaN,NaN,27388,NaN,NaN,642,9258,2539
2,201901014,2018-03-28,2018-03-30,887,1269,207,NaN,NaN,NaN,27388,NaN,NaN,2743,34760,136
3,201901015,2018-03-28,2018-03-30,887,1269,207,NaN,NaN,NaN,27388,1296312.0,1559462.0,37,19537,0
4,201901016,2018-03-28,2018-03-30,887,1269,207,NaN,NaN,NaN,27388,NaN,NaN,3469,42107,6003
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
34038974,2020092382684,2020-09-08,2020-09-11,1273,1269,1629,NaN,NaN,NaN,37472,675504.0,1997811.0,386,7370,0
34038975,2020092382685,2020-07-19,2020-07-20,712,1269,1288,NaN,NaN,NaN,27003,3294336.0,409782.0,338,9633,0
34038976,2020092382686,2020-07-19,2020-07-20,712,1269,1288,NaN,NaN,NaN,27003,3294336.0,409782.0,432,7906,0
34038977,2020092382687,2020-07-20,2020-07-20,716,1269,1623,NaN,NaN,NaN,32116,NaN,NaN,40,19750,99


In [37]:
dimCargoMeasurementDf

,identifier,manifest_unit,measurement_unit
0,201901012,PKG,Cubic Meters
1,201901013,PKG,Cubic Meters
2,201901014,CTN,Cubic Meters
3,201901015,PCL,Cubic Meters
4,201901016,CTN,Cubic Meters
...,...,...,...
34038974,2020092382684,CTN,NaN
34038975,2020092382685,CTN,NaN
34038976,2020092382686,CTN,NaN
34038977,2020092382687,PKG,Cubic Meters


---

# Insert Quantitative columns into fact_header from the dim_cargo_measurement table

In [27]:
import pandas as pd

def insertFactVariables(fact_header, dim_df, output_path):
    
    # Define columns added to fact_header
    measures = [
        "identifier",
        "manifest_quantity",
        "weight",
        "measurement"
    ]

    dim_subset = dim_df[measures]

    # Left Join fact_header with dim_subset
    newHeader = fact_header.merge(
        dim_subset,
        on="identifier",
        how="left",
        validate="one_to_one"
    )

    newHeader.to_csv(output_path, index=False)

    return


In [28]:
insertFactVariables(factHeaderDf, dimCargoMeasurementDf, "GoldLayerData/fact_header.csv")
log_transformation('GoldLayerData/fact_header.csv & /dim_cargo_measurement.csv', 'GoldLayerData/fact_header.csv', 'Add the quantitative variables into fact_header table from dim_cargo_measurement table', 'Denny Ye')

---

# Drop Unnecessary Columns for Use Case

In [32]:
def drop_columns(df, output_path, columns_to_drop):
    df.drop(columns=columns_to_drop, errors='ignore', inplace=True)
    df.to_csv(output_path, index=False)
    return df

- Dropping manifest_quantity, weight, and measurement from dim_cargo_measurement table

In [33]:
drop_columns(dimCargoMeasurementDf, 'GoldLayerData/dim_cargo_measurement.csv', ['manifest_quantity', 'weight', 'measurement'])
log_transformation('GoldLayerData/dim_cargo_measurement.csv', 'GoldLayerData/dim_cargo_measurement.csv', 'Drop manifest_quantity, weight, and measurement from dim_cargo_measurement table', 'Denny Ye')

- Dropping carrier_code, record_status_indicator, place_of_receipt, conveyance_id_qualifier, conveyance_id, in_bond_entry_type, mode_of_transportation from fact_header table

In [34]:
drop_columns(factHeaderDf, 'GoldLayerData/fact_header.csv', ['carrier_code', 'record_status_indicator', 'place_of_receipt', 'conveyance_id_qualifier', 'conveyance_id', 'in_bond_entry_type', 'mode_of_transportation'])
log_transformation('GoldLayerData/fact_header.csv', 'GoldLayerData/fact_header.csv', 'Drop carrier_code, record_status_indicator, place_of_receipt, conveyance_id_qualifier, conveyance_id, in_bond_entry_type, and mode_of_transportation from fact_header table', 'Denny Ye')